# External Event Triggers

This demo shows how to control megane's viewer from external tools (e.g., Plotly) via events.

- **Frame selection**: Click a Plotly data point to switch trajectory frames
- **Atom selection & measurement**: Select atoms from Python and retrieve distances, angles, and dihedrals
- **Event callbacks**: Subscribe to megane events to update other widgets


In [ ]:
import megane

print(f"megane v{megane.__version__}")

## 1. Plotly Integration: Click to Select Frame

Click a data point on the time-series plot (e.g., energy) to display
the corresponding trajectory frame in the viewer.


In [ ]:
import numpy as np

viewer = megane.MolecularViewer()
viewer.load("../tests/fixtures/caffeine_water.pdb", xtc="../tests/fixtures/caffeine_water_vibration.xtc")
print(f"Loaded trajectory: {viewer.total_frames} frames")

In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets

# Generate dummy energy data
n_frames = viewer.total_frames
frames = np.arange(n_frames)
energy = -500 + 10 * np.sin(frames * 0.1) + np.random.randn(n_frames) * 2

# Create Plotly FigureWidget
fig = go.FigureWidget(
    data=[go.Scatter(x=frames, y=energy, mode="lines+markers", name="Energy")],
    layout=go.Layout(
        title="Energy vs Frame (click a point to jump)",
        xaxis_title="Frame",
        yaxis_title="Energy (kJ/mol)",
        height=300,
    ),
)

# Connect click event to megane frame selection
def on_plotly_click(trace, points, state):
    if points.point_inds:
        viewer.frame_index = points.point_inds[0]

fig.data[0].on_click(on_plotly_click)

# Display stacked vertically
widgets.VBox([fig, viewer])

## 2. Programmatic Atom Selection and Measurement

Set `selected_atoms` to a list of atom indices to highlight atoms
in the viewer and automatically compute measurements.

| Atoms | Measurement |
|-------|-------------|
| 2 | Distance (Å) |
| 3 | Angle (°) |
| 4 | Dihedral (°) |


In [ ]:
v = megane.MolecularViewer()
v.load("../tests/fixtures/caffeine_water.pdb")
v

In [ ]:
# 2 atoms → distance
v.selected_atoms = [0, 1]
print("Distance:", v.measurement)

In [ ]:
# 3 atoms → angle
v.selected_atoms = [0, 1, 2]
print("Angle:", v.measurement)

In [ ]:
# 4 atoms → dihedral angle
v.selected_atoms = [0, 1, 2, 3]
print("Dihedral:", v.measurement)

In [ ]:
# Clear selection
v.selected_atoms = []

## 3. Event Callbacks

Use `on_event()` to subscribe to megane events.
Works as both a decorator and a method call.

| Event | Trigger | Data |
|-------|---------|------|
| `frame_change` | `frame_index` changes | `{"frame_index": int}` |
| `selection_change` | `selected_atoms` changes | `{"atoms": [int, ...]}` |
| `measurement` | Measurement result updated | `{"type", "value", "label", "atoms"}` |


In [ ]:
cb_viewer = megane.MolecularViewer()
cb_viewer.load("../tests/fixtures/caffeine_water.pdb", xtc="../tests/fixtures/caffeine_water_vibration.xtc")

# Register events with decorators
@cb_viewer.on_event("frame_change")
def on_frame(data):
    print(f"Frame changed: {data['frame_index']}")

@cb_viewer.on_event("measurement")
def on_meas(data):
    if data:
        print(f"Measurement: {data['type']} = {data['label']}")

cb_viewer

In [ ]:
# Change frame from Python → frame_change event fires
cb_viewer.frame_index = 10

In [ ]:
# Unregister callback
cb_viewer.off_event("frame_change", on_frame)
cb_viewer.frame_index = 20  # no longer prints

## 4. Bidirectional Sync: Update Plotly Marker on Frame Change

Use megane's frame change event to synchronize
a marker on the Plotly chart with the current frame position.


In [ ]:
sync_viewer = megane.MolecularViewer()
sync_viewer.load("../tests/fixtures/caffeine_water.pdb", xtc="../tests/fixtures/caffeine_water_vibration.xtc")

n = sync_viewer.total_frames
x = np.arange(n)
y = -500 + 10 * np.sin(x * 0.1) + np.random.randn(n) * 2

sync_fig = go.FigureWidget(
    data=[
        go.Scatter(x=x, y=y, mode="lines", name="Energy"),
        go.Scatter(x=[0], y=[y[0]], mode="markers", marker=dict(size=12, color="red"), name="Current"),
    ],
    layout=go.Layout(title="Bidirectional sync", height=300,
                     xaxis_title="Frame", yaxis_title="Energy (kJ/mol)"),
)

# Plotly → megane
def on_click(trace, points, state):
    if points.point_inds:
        sync_viewer.frame_index = points.point_inds[0]

sync_fig.data[0].on_click(on_click)

# megane → Plotly (update red marker)
@sync_viewer.on_event("frame_change")
def update_marker(data):
    idx = data["frame_index"]
    with sync_fig.batch_update():
        sync_fig.data[1].x = [idx]
        sync_fig.data[1].y = [y[idx]]

widgets.VBox([sync_fig, sync_viewer])

## 5. Dihedral Angle Trajectory Analysis

Compute the dihedral angle at each frame and plot the results with Plotly.
Combines `frame_index` and `selected_atoms`.


In [ ]:
import time

dih_viewer = megane.MolecularViewer()
dih_viewer.load("../tests/fixtures/caffeine_water.pdb", xtc="../tests/fixtures/caffeine_water_vibration.xtc")

# Set atoms for dihedral measurement
dih_viewer.selected_atoms = [0, 1, 2, 3]

# Collect dihedral at each frame
dihedrals = []

@dih_viewer.on_event("measurement")
def collect_dihedral(data):
    if data and data["type"] == "dihedral":
        dihedrals.append(data["value"])

for i in range(dih_viewer.total_frames):
    dih_viewer.frame_index = i
    time.sleep(0.01)  # wait for widget sync

dih_viewer.off_event("measurement", collect_dihedral)

# Plot the results
dih_fig = go.FigureWidget(
    data=[go.Scatter(x=list(range(len(dihedrals))), y=dihedrals, mode="lines+markers")],
    layout=go.Layout(
        title="Dihedral angle along trajectory",
        xaxis_title="Frame",
        yaxis_title="Dihedral (°)",
        height=300,
    ),
)

widgets.VBox([dih_fig, dih_viewer])

## 6. Custom Widget Integration

Connect any widget (e.g., `ipywidgets` sliders) with `on_event` for seamless integration.


In [ ]:
slider_viewer = megane.MolecularViewer()
slider_viewer.load("../tests/fixtures/caffeine_water.pdb", xtc="../tests/fixtures/caffeine_water_vibration.xtc")

slider = widgets.IntSlider(
    value=0, min=0, max=slider_viewer.total_frames - 1,
    description="Frame:", continuous_update=True,
)
label = widgets.Label(value="Frame: 0")

# Slider → megane
def on_slider(change):
    slider_viewer.frame_index = change["new"]

slider.observe(on_slider, names="value")

# megane → label (label updates even when using the browser timeline)
@slider_viewer.on_event("frame_change")
def on_frame_for_label(data):
    idx = data["frame_index"]
    label.value = f"Frame: {idx}"
    slider.value = idx  # Sync slider too

widgets.VBox([widgets.HBox([slider, label]), slider_viewer])